In [140]:
# Bibliotecas de Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import pandas as pd

In [141]:
df = pd.read_csv("../data/ChurnCleanDataset.csv")

df

,SeniorCitizen,Partner,Dependents,tenure,MonthlyCharges,TotalCharges,Churn,MultipleLines_No,MultipleLines_No phone service,MultipleLines_Yes,...,Contract_Two year,PaperlessBilling_No,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,DeviceProtection_No,DeviceProtection_No internet service,DeviceProtection_Yes
0,0,1,0,1,29.85,29.85,0,0,1,0,...,0,0,1,0,0,1,0,1,0,0
1,0,0,0,34,56.95,1889.50,0,1,0,0,...,0,1,0,0,0,0,1,0,0,1
2,0,0,0,2,53.85,108.15,1,1,0,0,...,0,0,1,0,0,0,1,1,0,0
3,0,0,0,45,42.30,1840.75,0,0,1,0,...,0,1,0,1,0,0,0,0,0,1
4,0,0,0,2,70.70,151.65,1,1,0,0,...,0,0,1,0,0,1,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7027,0,1,1,24,84.80,1990.50,0,0,0,1,...,0,0,1,0,0,0,1,0,0,1
7028,0,1,1,72,103.20,7362.90,0,0,0,1,...,0,0,1,0,1,0,0,0,0,1
7029,0,1,1,11,29.60,346.45,0,0,1,0,...,0,0,1,0,0,1,0,1,0,0
7030,1,1,0,4,74.40,306.60,1,0,0,1,...,0,0,1,0,0,0,1,1,0,0


In [142]:
# Colocando a variável dependente na última posição do dataset
features = [col for col in df.columns if col != 'Churn']
features_target = features + ['Churn']
df = df[features_target]


# Definindo as nossas features (X) e o target (y)

X = df.iloc[:, :-1] # todas as colunas menos a última
y = df.iloc[:, -1] # última coluna

df.head()

,SeniorCitizen,Partner,Dependents,tenure,MonthlyCharges,TotalCharges,MultipleLines_No,MultipleLines_No phone service,MultipleLines_Yes,InternetService_DSL,...,PaperlessBilling_No,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,DeviceProtection_No,DeviceProtection_No internet service,DeviceProtection_Yes,Churn
0,0,1,0,1,29.85,29.85,0,1,0,1,...,0,1,0,0,1,0,1,0,0,0
1,0,0,0,34,56.95,1889.50,1,0,0,1,...,1,0,0,0,0,1,0,0,1,0
2,0,0,0,2,53.85,108.15,1,0,0,1,...,0,1,0,0,0,1,1,0,0,1
3,0,0,0,45,42.30,1840.75,0,1,0,1,...,1,0,1,0,0,0,0,0,1,0
4,0,0,0,2,70.70,151.65,1,0,0,0,...,0,1,0,0,1,0,1,0,0,1


In [143]:
# Checar os dados, plotar os gráficos necessários (HEATMAP!!!), boxplot, ver as correlações
# Decidir quais features manter, quais descartar
df.head()
df.describe()
df.info()
df.corr()

# Por enquanto vou manter todas e após o treinamento do modelo vou descartando para uma possível melhora da acurácia

<class 'pandas.DataFrame'>
RangeIndex: 7032 entries, 0 to 7031
Data columns (total 40 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   SeniorCitizen                            7032 non-null   int64  
 1   Partner                                  7032 non-null   int64  
 2   Dependents                               7032 non-null   int64  
 3   tenure                                   7032 non-null   int64  
 4   MonthlyCharges                           7032 non-null   float64
 5   TotalCharges                             7032 non-null   float64
 6   MultipleLines_No                         7032 non-null   int64  
 7   MultipleLines_No phone service           7032 non-null   int64  
 8   MultipleLines_Yes                        7032 non-null   int64  
 9   InternetService_DSL                      7032 non-null   int64  
 10  InternetService_Fiber optic              7032 non-null   in

,SeniorCitizen,Partner,Dependents,tenure,MonthlyCharges,TotalCharges,MultipleLines_No,MultipleLines_No phone service,MultipleLines_Yes,InternetService_DSL,...,PaperlessBilling_No,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,DeviceProtection_No,DeviceProtection_No internet service,DeviceProtection_Yes,Churn
SeniorCitizen,1.000000,0.016957,-0.210550,0.015683,0.219874,0.102411,-0.136377,-0.008392,0.142996,-0.108276,...,-0.156258,0.156258,-0.016235,-0.024359,0.171322,-0.152987,0.094403,-0.182519,0.059514,0.150541
Partner,0.016957,1.000000,0.452269,0.381912,0.097825,0.319072,-0.130028,-0.018397,0.142561,-0.001043,...,0.013957,-0.013957,0.111406,0.082327,-0.083207,-0.096948,-0.146702,-0.000286,0.153556,-0.149982
Dependents,-0.210550,0.452269,1.000000,0.163386,-0.112343,0.064653,0.023388,0.001078,-0.024307,0.051593,...,0.110131,-0.110131,0.052369,0.061134,-0.149274,0.056448,-0.128053,0.138383,0.013900,-0.163128
tenure,0.015683,0.381912,0.163386,1.000000,0.246862,0.825880,-0.323891,-0.007877,0.332399,0.013786,...,-0.004823,0.004823,0.243822,0.232800,-0.210197,-0.232181,-0.314820,-0.037529,0.361520,-0.354049
MonthlyCharges,0.219874,0.097825,-0.112343,0.246862,1.000000,0.651065,-0.338514,-0.248033,0.490912,-0.161368,...,-0.351930,0.351930,0.042410,0.030055,0.271117,-0.376568,0.171057,-0.763191,0.482607,0.192858
TotalCharges,0.102411,0.319072,0.064653,0.825880,0.651065,1.000000,-0.396765,-0.113008,0.469042,-0.052190,...,-0.157830,0.157830,0.186119,0.182663,-0.060436,-0.294708,-0.189485,-0.374878,0.522881,-0.199484
MultipleLines_No,-0.136377,-0.130028,0.023388,-0.323891,-0.338514,-0.396765,1.000000,-0.315218,-0.823076,-0.069515,...,0.151974,-0.151974,-0.069663,-0.063712,-0.080990,0.222395,-0.026582,0.309984,-0.240847,-0.032654
MultipleLines_No phone service,-0.008392,-0.018397,0.001078,-0.007877,-0.248033,-0.113008,-0.315218,1.000000,-0.279530,0.452255,...,0.016696,-0.016696,-0.008271,0.006916,-0.002747,0.004463,0.075421,-0.171817,0.070076,-0.011691
MultipleLines_Yes,0.142996,0.142561,-0.024307,0.332399,0.490912,0.469042,-0.823076,-0.279530,1.000000,-0.200318,...,-0.163746,0.163746,0.075429,0.060319,0.083583,-0.227672,-0.018242,-0.210794,0.201733,0.040033
InternetService_DSL,-0.108276,-0.001043,0.051593,0.013786,-0.161368,-0.052190,-0.069515,0.452255,-0.200318,1.000000,...,0.063390,-0.063390,0.024760,0.051222,-0.104293,0.042754,0.176142,-0.379912,0.145150,-0.124141


In [144]:
# Visualizando as nossas features

print("=== FEATURES (X) ===")
print(f"Formato do dataset: {X.shape}")
print(f"Número de amostras: {X.shape[0]}")
print(f"Número de features: {X.shape[1]}")

=== FEATURES (X) ===
Formato do dataset: (7032, 39)
Número de amostras: 7032
Número de features: 39


In [145]:
# Visualizando o nosso target


# Transformando a flag binária da coluna 'TenYearCHD' em strings
nomeando_classes = {0: 'Sem Doença Cardíaca', 1: 'Com Doença Cardíaca'}
y_classes = y.map(nomeando_classes)


print(f"Classes do target: {y_classes.unique()}")
print(f"Distribuição das classes:")
print(f"  Sem Doença Cardíaca (0): {sum(y == 0)} casos ({sum(y == 0)/len(y)*100:.1f}%)")
print(f"  Com Doença Cardíaca (1): {sum(y == 1)} casos ({sum(y == 1)/len(y)*100:.1f}%)")

Classes do target: <StringArray>
['Sem Doença Cardíaca', 'Com Doença Cardíaca']
Length: 2, dtype: str
Distribuição das classes:
  Sem Doença Cardíaca (0): 5163 casos (73.4%)
  Com Doença Cardíaca (1): 1869 casos (26.6%)


In [146]:
linhas_com_nulos = df[df.isnull().any(axis=1)]

print(linhas_com_nulos)

Empty DataFrame
Columns: [SeniorCitizen, Partner, Dependents, tenure, MonthlyCharges, TotalCharges, MultipleLines_No, MultipleLines_No phone service, MultipleLines_Yes, InternetService_DSL, InternetService_Fiber optic, InternetService_No, OnlineSecurity_No, OnlineSecurity_No internet service, OnlineSecurity_Yes, OnlineBackup_No, OnlineBackup_No internet service, OnlineBackup_Yes, TechSupport_No, TechSupport_No internet service, TechSupport_Yes, StreamingTV_No, StreamingTV_No internet service, StreamingTV_Yes, StreamingMovies_No, StreamingMovies_No internet service, StreamingMovies_Yes, Contract_Month-to-month, Contract_One year, Contract_Two year, PaperlessBilling_No, PaperlessBilling_Yes, PaymentMethod_Bank transfer (automatic), PaymentMethod_Credit card (automatic), PaymentMethod_Electronic check, PaymentMethod_Mailed check, DeviceProtection_No, DeviceProtection_No internet service, DeviceProtection_Yes, Churn]
Index: []

[0 rows x 40 columns]


In [147]:
# Informações sobre o nosso dataset
print("=== VISÃO GERAL DO DATASET ===")
print(f"Formato do dataset: {X.shape}")
print(f"Formato do target: {y.shape}")


print("\n=== PRIMEIRAS 5 LINHAS ===")
print(X.head())


print("\n=== RESUMO ESTATÍSTICO ===")
print(X.describe())


print("\n=== TIPOS DE DADOS E VALORES AUSENTES ===")
print(X.info())
print(f"\nValores ausentes nas features: {X.isnull().sum().sum()}")
print(f"Valores ausentes no target: {pd.isnull(y).sum()}")

=== VISÃO GERAL DO DATASET ===
Formato do dataset: (7032, 39)
Formato do target: (7032,)

=== PRIMEIRAS 5 LINHAS ===
   SeniorCitizen  Partner  Dependents  tenure  MonthlyCharges  TotalCharges  \
0              0        1           0       1           29.85         29.85   
1              0        0           0      34           56.95       1889.50   
2              0        0           0       2           53.85        108.15   
3              0        0           0      45           42.30       1840.75   
4              0        0           0       2           70.70        151.65   

   MultipleLines_No  MultipleLines_No phone service  MultipleLines_Yes  \
0                 0                               1                  0   
1                 1                               0                  0   
2                 1                               0                  0   
3                 0                               1                  0   
4                 1                   

In [148]:
# Separar treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [149]:
# Fazer o Feature Scaling (se necessário)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Features standardized")

Features standardized


In [150]:
# Criar o modelo (usando uma classe do sklearn)
# Nesse exemplo é Regressão Logística, mas pode ser a Linear que vimos
# Ou qualquer outro modelo que você quiser (e.g. SVM, Random Forest, etc)
model = LogisticRegression(random_state=42)

In [151]:
# Treinar o modelo (sim, o fit faz todo aquele processo que vimos em sala)
# Ele vai calcular o erro, o gradiente, e atualizar os pesos
# Isso vai criar o modelo, e ele vai ser capaz de fazer previsões (criando a função de inferência)
model.fit(X_train_scaled, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [152]:
# Fazer o teste e as previsões necessárias
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)

In [153]:
# Fazer a avaliação do modelo
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.8045
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1033
           1       0.65      0.58      0.61       374

    accuracy                           0.80      1407
   macro avg       0.75      0.73      0.74      1407
weighted avg       0.80      0.80      0.80      1407

[[916 117]
 [158 216]]


In [154]:
print(f"\n=== CONCLUSÃO DA ANÁLISE ===")
if accuracy > 0.95:
    print("🎯 EXCELENTE: Modelo demonstra performance excepcional!")
elif accuracy > 0.90:
    print("✅ MUITO BOM: Modelo apresenta alta acurácia!")
elif accuracy > 0.85:
    print("👍 BOM: Modelo apresenta boa performance!")
else:
    print("⚠️  MELHORIAS NECESSÁRIAS: Modelo precisa de ajustes!")


=== CONCLUSÃO DA ANÁLISE ===
⚠️  MELHORIAS NECESSÁRIAS: Modelo precisa de ajustes!
